In [2]:
import pandas as pd
import pickle
import numpy as np

In [3]:
import pandas as pd
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# 1. Load Data
try:
    df_meal = pd.read_csv('../../datas/dataset_meal.csv')
    print(f"Data Makanan Dimuat: {len(df_meal)} menu.")
except:
    print("File dataset_meal.csv tidak ditemukan.")
    # Buat data dummy biar kode tetap jalan saat dicoba
    data_dummy = {
        'Food Items': ['Grilled Cheese', 'Pizza', 'Oatmeal', 'Eggs', 'Protein Shake'],
        'Energy kcal': [400, 300, 150, 80, 120],
        'Protein(g)': [12, 10, 5, 6, 25],
        'Carbs': [30, 40, 25, 1, 5],
        'Fat(g)': [25, 12, 3, 5, 1]
    }
    df_meal = pd.DataFrame(data_dummy)

# 2. Preprocessing
features = ['Energy kcal', 'Protein(g)', 'Carbs', 'Fat(g)']
df_meal_clean = df_meal.dropna(subset=features).reset_index(drop=True)

# PENTING: Kita pakai StandardScaler agar AI melihat 'Rasio/Kualitas', bukan besar-kecilnya angka.
# Jadi dia bisa tahu bahwa "Telur 1 butir" itu mirip dengan "Telur 3 butir" secara nutrisi.
scaler = StandardScaler()
meal_features_scaled = scaler.fit_transform(df_meal_clean[features])

# 3. Latih Model KNN
# Kita cari makanan yang profil nutrisinya mirip
knn_meal = NearestNeighbors(n_neighbors=10, metric='euclidean')
knn_meal.fit(meal_features_scaled)

print("Model KNN Meal berhasil dilatih!")

# 4. Simpan ke Pickle
data_meal = {
    'knn_model': knn_meal,
    'scaler': scaler,      # Simpan scaler-nya juga!
    'meal_db': df_meal_clean,
    'features': features
}

with open('model_meal.pickle', 'wb') as f:
    pickle.dump(data_meal, f)

print("SUKSES: 'model_meal.pickle' telah disimpan.")

Data Makanan Dimuat: 1028 menu.
Model KNN Meal berhasil dilatih!
SUKSES: 'model_meal.pickle' telah disimpan.


In [7]:
import pickle
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Load Model
try:
    with open('model_meal.pickle', 'rb') as f:
        meal_data = pickle.load(f)
    knn = meal_data['knn_model']
    scaler = meal_data['scaler']
    db_meal = meal_data['meal_db']
    features = meal_data['features']
    print("✅ Model Meal siap digunakan.")
except:
    print("❌ Error: Pickle tidak ditemukan.")
    exit()

# --- FUNGSI GENERATE JADWAL SEHARIAN ---
def generate_full_day_plan(target_harian, freq_makan):
    
    # 1. Hitung Target Per Sekali Makan (Bagi Rata)
    target_per_meal = {
        'Energy kcal': target_harian['Daily_Calories'] / freq_makan,
        'Protein(g)': target_harian['Target_Protein_g'] / freq_makan,
        'Carbs': target_harian['Target_Carbs_g'] / freq_makan,
        'Fat(g)': target_harian['Target_Fat_g'] / freq_makan
    }
    
    print(f"🎯 TARGET HARIAN: {target_harian['Daily_Calories']} kkal")
    print(f"🍽️ FREKUENSI: {freq_makan}x makan (Per sesi: ~{target_per_meal['Energy kcal']:.0f} kkal)")
    print("-" * 60)
    
    # List untuk menyimpan menu yang sudah terpilih agar tidak kembar
    menu_sudah_dipilih = []
    
    # Siapkan data input untuk AI (DataFrame biar ga warning)
    input_df = pd.DataFrame([
        [
            target_per_meal['Energy kcal'], 
            target_per_meal['Protein(g)'], 
            target_per_meal['Carbs'], 
            target_per_meal['Fat(g)']
        ]
    ], columns=features)
    
    input_scaled = scaler.transform(input_df)
    
    # 2. LOOPING UNTUK SETIAP SESI MAKAN
    for i in range(1, freq_makan + 1):
        
        # Cari kandidat agak banyak (misal 15) buat cadangan kalau yang atas udah kepilih
        distances, indices = knn.kneighbors(input_scaled, n_neighbors=15)
        candidates = db_meal.iloc[indices[0]].copy()
        
        # Filter: Buang menu yang sudah ada di list 'menu_sudah_dipilih'
        candidates = candidates[~candidates['Food Items'].isin(menu_sudah_dipilih)]
        
        # Jika kandidat habis (jarang terjadi), reset list
        if candidates.empty:
            candidates = db_meal.iloc[indices[0]].copy()
        
        # Ambil Top 1 terbaik untuk jam makan ini
        pilihan = candidates.iloc[0]
        
        # Hitung Porsi (Bulatkan ke 0.5 terdekat)
        porsi = target_per_meal['Energy kcal'] / pilihan['Energy kcal']
        porsi = round(porsi * 2) / 2
        if porsi < 0.5: porsi = 0.5 # Minimal setengah porsi
        
        # Simpan nama menu ini biar ga muncul di jam makan berikutnya
        menu_sudah_dipilih.append(pilihan['Food Items'])
        
        # --- TAMPILKAN HASIL ---
        kalori_masuk = pilihan['Energy kcal'] * porsi
        protein_masuk = pilihan['Protein(g)'] * porsi
        
        print(f"⏰ MAKAN KE-{i}:")
        print(f"   Menu    : {pilihan['Food Items']}")
        print(f"   Porsi   : {porsi} porsi")
        print(f"   Nutrisi : {kalori_masuk:.0f} kkal | {protein_masuk:.1f}g Protein")
        print("-" * 30)

# --- CONTOH PEMAKAIAN ---

# Data dari User Progress (AI LSTM kamu)
user_target = {
    'Daily_Calories': 2750,
    'Target_Protein_g': 205,
    'Target_Carbs_g': 324,
    'Target_Fat_g': 57
}

jumlah_makan = 5

generate_full_day_plan(user_target, jumlah_makan)

✅ Model Meal siap digunakan.
🎯 TARGET HARIAN: 2750 kkal
🍽️ FREKUENSI: 5x makan (Per sesi: ~550 kkal)
------------------------------------------------------------
⏰ MAKAN KE-1:
   Menu    : Gun powder chutney
   Porsi   : 2.0 porsi
   Nutrisi : 625 kkal | 43.1g Protein
------------------------------
⏰ MAKAN KE-2:
   Menu    : Maa chaane ki dal
   Porsi   : 1.5 porsi
   Nutrisi : 517 kkal | 29.7g Protein
------------------------------
⏰ MAKAN KE-3:
   Menu    : Lobster Roll Sandwich
   Porsi   : 1.0 porsi
   Nutrisi : 450 kkal | 20.0g Protein
------------------------------
⏰ MAKAN KE-4:
   Menu    : Bengal 5 Spice Blend (Panch Phoran)
   Porsi   : 2.0 porsi
   Nutrisi : 580 kkal | 36.5g Protein
------------------------------
⏰ MAKAN KE-5:
   Menu    : Cracked wheat and green gram dal premix (Dalia moong dal premix)
   Porsi   : 1.5 porsi
   Nutrisi : 543 kkal | 23.8g Protein
------------------------------
